In [2]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# LangChain에서 채팅용 언어 모델(LLM)을 초기화하는 함수입니다.
from langchain.chat_models import init_chat_model

# 대화 메시지(history)를 상태로 다룰 수 있게 해주는 기본 상태 타입입니다.
from langgraph.graph.message import MessagesState

# 실제로 사용할 채팅 모델(LLM)을 초기화합니다.
# - "openai: gpt-4o-mini" 라는 이름의 모델을 사용하도록 설정되어 있습니다.
# - 이 코드를 실행하려면 .env 등에 OPENAI_API_KEY 와 같은 환경 변수가 설정되어 있어야 합니다.
llm = init_chat_model("openai: gpt-4o-mini")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# MessagesState 를 상속해서, 대화형 상태를 위한 State 를 정의합니다.
# - MessagesState 안에는 보통 "messages" 라는 키에 대화 내역이 들어갑니다.
# - 여기에 custom_stuff 같은 필드를 추가해서, 필요하면 더 많은 상태를 담을 수 있습니다.
class State(MessagesState):
  custom_stuff: str


# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
graph_builder = StateGraph(State)

In [ ]:
# chatbot 노드는 현재까지의 대화 메시지(state["messages"])를
# LLM 에게 보내고, 그 응답을 다시 messages 에 추가해서 반환합니다.
# - state["messages"]: 지금까지의 대화 내역 (user/assistant 메시지들)
# - llm.invoke(...): 이 대화 내역을 모델에 전달해 답변을 생성
# - 반환: 새로 생성된 응답 메시지를 리스트 형태로 돌려줌
#   (LangGraph 가 기존 messages 와 합쳐서 전체 대화 내역을 관리합니다.)
def chatbot(state: State):
  response = llm.invoke(state["messages"])
  return {
    "messages": [response]
  }

In [ ]:
# 그래프에 chatbot 노드를 등록합니다.
graph_builder.add_node("chatbot", chatbot)

# START 에서 chatbot 으로, chatbot 에서 END 로 이어지는
# 아주 단순한 그래프 구조입니다.
# START -> chatbot -> END
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# 지금까지 정의한 노드와 간선을 이용해서
# 실제로 실행 가능한 그래프 객체를 만듭니다.
graph = graph_builder.compile()

# 그래프를 한 번 실행해 봅니다.
# - 초기 상태로 messages 에 user 의 질문을 하나 넣어 줍니다.
# - 흐름: START → chatbot
#   * chatbot 이 messages 를 llm 에 보내 답을 생성
#   * 생성된 응답을 messages 에 추가한 뒤 END 로 이동
# - 최종적으로 user 질문과 모델 응답이 함께 들어 있는 messages 상태를 얻습니다.
graph.invoke(
  {
    "messages": [
      {"role": "user", "content": "How are you?"},
    ]
  }
)

### 참고

- 이 예제는 LangGraph + LangChain 을 이용해서
  "한 번만 대화를 주고받는" 아주 간단한 챗봇 그래프입니다.
- 실제로 여러 턴(turn)의 대화를 이어가고 싶다면,
  그래프 구조를 반복 호출하거나, 노드를 더 추가해
  상태를 계속 이어가는 패턴을 사용할 수 있습니다.
- 또한 이 노트북을 실행하려면 OpenAI API 키 등이
  `.env` 나 환경 변수에 올바르게 설정되어 있어야 합니다.